In [1]:
import sys
sys.path.append("..")
from kg.relation_extractor import RelationExtractor
import pandas as pd
import spacy

nlp = spacy.load("en_core_sci_sm")
rel_ext = RelationExtractor()
entities_df = pd.read_csv("../data/processed/entities.csv")

triples = []
sample_file = entities_df["source_file"].iloc[0]
with open(f"../data/raw/{sample_file}", "r") as f:
    text = f.read()
doc = nlp(text)

for sent in doc.sents:
    ents_in_sent = []
    for _, row in entities_df[entities_df["source_file"] == sample_file].iterrows():
        if row["start"] >= sent.start_char and row["end"] <= sent.end_char:
            ents_in_sent.append(row)
    if len(ents_in_sent) >= 2:
        for i in range(len(ents_in_sent)):
            for j in range(i+1, len(ents_in_sent)):
                rel = rel_ext.extract_relation(sent.text, ents_in_sent[i]["text"], ents_in_sent[j]["text"])
                triples.append({
                    "head": ents_in_sent[i]["text"],
                    "head_type": ents_in_sent[i]["node_type"],
                    "relation": rel,
                    "tail": ents_in_sent[j]["text"],
                    "tail_type": ents_in_sent[j]["node_type"],
                    "sentence": sent.text,
                    "source_file": sample_file
                })

pd.DataFrame(triples).to_csv("../data/processed/triples.csv", index=False)
print(f"Extracted {len(triples)} triples")

d:\DCE-KG\venv\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Extracted 419 triples
